# SuperNEMO Transformer results

Loads ignored outputs only after confirming that all six runs share the split, EnergyBench protocol, evaluator, and prediction ordering. This notebook does not write a result file by default.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'supernemo_detector').is_dir() and (candidate / 'pyproject.toml').is_file()
)
OUTPUT_ROOT = PROJECT_ROOT / 'supernemo_detector' / 'outputs' / 'classification'
MODEL_IDS = (
    'transformer_001_entity_coordinate_mlp',
    'transformer_002_patch_88mm_coordinate_mlp',
    'transformer_003_patch_88mm_fourier_xyz',
    'transformer_004_entity_fourier_xyz',
    'transformer_005_summary_16_coordinate_mlp',
    'transformer_006_summary_16_fourier_xyz',
)

In [ ]:
def read_json(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding='utf-8'))

records = []
identities = {}
for model_id in MODEL_IDS:
    run_dir = OUTPUT_ROOT / model_id
    config = read_json(run_dir / 'run_config.json')
    metrics = read_json(run_dir / 'test_evaluation' / 'test_metrics.json')
    provenance = config['dataset_provenance']
    identities[model_id] = {
        'split': provenance['manifest_content_sha256'],
        'evaluation_manifest': provenance['evaluation_protocol']['manifest_sha256'],
        'evaluator': provenance['evaluation_protocol']['evaluator_code_sha256'],
        'validation_order': provenance['prediction_order_sha256']['validation'],
        'test_order': provenance['prediction_order_sha256']['test'],
    }
    records.append({
        'model_id': model_id,
        'representation': config['tokenization']['tokenization'],
        'position_encoding': config['model']['position_encoding'],
        'energy_matched_auc': metrics['energy_matched_auc'],
        'auc': metrics['auc'],
        'accuracy': metrics['accuracy'],
        'events': metrics['events'],
        'coverage_minimum': metrics.get('token_coverage_minimum'),
    })

In [ ]:
reference_id = MODEL_IDS[0]
reference = identities[reference_id]
mismatches = {
    model_id: {key: (reference[key], value) for key, value in identity.items() if value != reference[key]}
    for model_id, identity in identities.items()
}
mismatches = {model_id: values for model_id, values in mismatches.items() if values}
if mismatches:
    raise ValueError(f'Cannot combine runs with different provenance: {mismatches}')

results = pd.DataFrame.from_records(records).sort_values(
    'energy_matched_auc', ascending=False
).reset_index(drop=True)
results